In [1]:
import numpy as np
import bilby
import detection

In [2]:
# Create the injection using bilby
priors = bilby.core.prior.PriorDict()

# Intrinsic parameters
priors['mass_1'] = bilby.core.prior.analytical.PowerLaw(minimum=5, maximum=50, alpha=-2.3,
                                                        name='mass_1')
priors['mass_2'] = bilby.core.prior.analytical.PowerLaw(minimum=5, maximum=50, alpha=-2.3,
                                                        name='mass_2')
priors['a_1'] = bilby.core.prior.Uniform(name='a_1', minimum=0, maximum=0.99)
priors['a_2'] = bilby.core.prior.Uniform(name='a_2', minimum=0, maximum=0.99)
priors['tilt_1'] = bilby.core.prior.analytical.Sine(name='tilt_1')
priors['tilt_2'] = bilby.core.prior.analytical.Sine(name='tilt_2')
priors['phi_12'] = bilby.core.prior.Uniform(name='phi_12', minimum=0,
                                            maximum=2 * np.pi, boundary='periodic')
priors['phi_jl'] = bilby.core.prior.Uniform(name='phi_jl', minimum=0,
                                            maximum=2 * np.pi, boundary='periodic')

# Extrinsic parameters
priors['phase'] = bilby.core.prior.Uniform(name='phase', minimum=0,
                                           maximum=2 * np.pi, boundary='periodic')
priors['theta_jn'] = bilby.core.prior.analytical.Sine(name='theta_jn')
priors['ra'] = bilby.core.prior.Uniform(name='ra', minimum=0, maximum=2 * np.pi, boundary='periodic')
priors['dec'] = bilby.core.prior.Cosine(name='dec')
priors['psi'] = bilby.core.prior.Uniform(name='psi', minimum=0, maximum=np.pi, boundary='periodic')
priors['luminosity_distance'] = bilby.gw.prior.UniformSourceFrame(name='luminosity_distance', minimum=1e2, maximum=5e3)

# Sample the injection from the priors
injection = priors.sample(1)
gwbench_injection = detection.build_gwbench_injection_from_bilby_params(
    injection["mass_1"][0], injection["mass_2"][0], 
    injection["a_1"][0], injection["a_2"][0],
    injection["tilt_1"][0], injection["tilt_2"][0], 
    injection["phi_12"][0], injection["phi_jl"][0],
    injection["theta_jn"][0], injection["luminosity_distance"][0], 
    injection["ra"][0], injection["dec"][0],
    injection["psi"][0], fref=-1, phase=injection["phase"][0])

In [3]:
# Specify the detector networks to analyze
# Given in the format "{detector}_{location}". 
# see: https://gitlab.com/sborhanian/gwbench/-/blob/master/example_scripts/README.md
network_specs = [
    ['CE-40_CEA'],
    ['CE-40_CEA', 'ET_ET1'],
    ['CE-40_CEA', 'CE-20_CEB', 'ET_ET1']
]

# Create tags for the different detector networks
tag_map = {
    tuple(['CE-40_CEA']): 'CE40',
    tuple(['CE-40_CEA', 'ET_ET1']): 'CE40ET',
    tuple(['CE-40_CEA', 'CE-20_CEB', 'ET_ET1']): 'CE40CE20ET'
}

out_data = detection.run_fisher_analysis(
    approximant = 'IMRPhenomXPHM',
    network_specs = network_specs,
    injection_parameters=gwbench_injection,
    tag_map = tag_map,
)

In [5]:
out_data["CE40"]

{'snr': np.float64(107.63320459806364),
 'errs': {'log_Mc': np.float32(1.4977749e-05),
  'eta': np.float32(1.4286743e-05),
  'chi1z': np.float32(0.0041487627),
  'chi2z': np.float32(0.004600895),
  'log_DL': np.float32(3.7097824),
  'tc': np.float32(0.029710485),
  'ra': np.float32(1.2473584),
  'cos_dec': np.float32(0.12925586),
  'psi': np.float32(1.3290029),
  'sky_area_90': np.float64(116722.09566587141)},
 'cov': matrix([[ 2.2433297e-10, -5.0257722e-11,  1.2069782e-08, -1.3987204e-09,
          -2.6030848e-05, -1.7893636e-07,  3.3652664e-06,  9.1328678e-07,
           5.8324686e-06],
         [-5.0257722e-11,  2.0411102e-10, -1.0520025e-08,  1.1259080e-08,
           1.3742484e-05,  9.2904280e-08, -1.8800537e-06, -4.8192925e-07,
          -2.9959772e-06],
         [ 1.2069782e-08, -1.0520025e-08,  1.7212231e-05, -1.8618954e-05,
          -1.5652302e-05,  3.4967235e-07,  5.1247054e-05,  1.1340995e-06,
           8.4258791e-05],
         [-1.3987204e-09,  1.1259080e-08, -1.8618954e-

In [ ]:
# define the cosmology to use
#cosmo = apcosmo.Planck18
#injection["redshift"] = apcosmo.z_at_value(cosmo.luminosity_distance, injection["luminosity_distance"] * u.Mpc).value